In [ ]:
# Human-in-the-loop (HITL) agentic AI is an architectural pattern where autonomous AI agents pause their automated workflows to seek human approval, guidance, or verification before proceeding with a task. It combines the speed of AI automation with the oversight, ethics, and accountability of human experts.

                #         START
                #           |
                #           |
                #       Draft Mail
                #           |
                #           |
                #     Human Feedback
                #         /\
                #        /  \
                #      Yes   NO ---------> Retry
                #       |     
                #       |
                #   Send Mail
                #       |
                #       |
                #      END

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
)

In [ ]:
class FlowState(BaseModel):
    query: str = "";
    draft: str = ""
    human_fd: str = ""
    final_res: str = ""

In [ ]:
from langgraph.types import interrupt, Command
from typing import Literal

# Nodes:
def draftmail_node(state: FlowState) -> FlowState:
    if state.human_fd:
        prompt = f"""
                  Rewrite this email draft based on the human feedback.

                  Query: {state.query}
                  
                  Original draft:
                  {state.draft}

                  Human feedback:
                  {state.human_fd}
                 """
                
        res = llm.invoke(prompt).content
        state.human_fd = ""  # clear feedback after using it
    else:
        res = llm.invoke(state.query).content

    state.draft = res
    return state


def human_feedback_node(state: FlowState) -> FlowState:
    fd = interrupt({
        "draft_mail": state.draft,
        "question": "Do you want to continue or re-write the mail?"
    })

    feedback = str(fd or "").strip().lower()

    if feedback in {"ok", "okay", "continue", "approved", "done", "yes"}:
        state.human_fd = ""
    else:
        state.human_fd = feedback

    return state


def final_node(state: FlowState) -> FlowState:
    state.final_res = state.draft
    print("mail sent successfully...")
    return state


def conditional_node(state: FlowState) -> Literal["draftmail_node", "final_node"]:
    if state.human_fd:
        return "draftmail_node"
    return "final_node"

In [ ]:


graph = StateGraph(FlowState)
graph.add_node("draftmail_node", draftmail_node)
graph.add_node("human_feedback_node", human_feedback_node)
graph.add_node("final_node", final_node)


graph.add_edge(START, "draftmail_node")
graph.add_edge("draftmail_node", "human_feedback_node")
graph.add_conditional_edges(
    "human_feedback_node",
    conditional_node,
)
graph.add_edge("final_node", END)

memory = InMemorySaver()
graph = graph.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
thread_config = {
    "configurable": {
        "thread_id": "loop-1"
    }
}

res = graph.invoke({
    "query":"Write an email to abc@gmail.com about the sick leave for 3days."
},
    config=thread_config
)

res

In [ ]:
res = graph.invoke(Command(resume="No, please re-write"), config=thread_config)
res

In [ ]:
res = graph.invoke(Command(resume="yes"), config=thread_config)
res